In [11]:
import zarr
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd

import dask.array as da
import dask.dataframe as dd

import datashader as ds
import colorcet as cc
import pvdeg

import holoviews as hv

In [2]:
live_data_copy = Path("/projects/inspire/PySAM-MAPS/v1.2/")

data_dir = live_data_copy / "final-backup/"
gid_file = live_data_copy / "gid-lat-lon.csv"

In [37]:
gids_mapping_df = pd.read_csv(gid_file, index_col=0)
gids_mapping_df.index.name = "gid"

#gids_mapping_df = ddf.from_pandas(gids_mapping_df) # might not be worth it to do this 
gids_mapping_df

,latitude,longitude
gid,,
0,-15.95,-179.98
1,-15.99,-179.98
2,-16.03,-179.98
3,-16.07,-179.98
4,-16.11,-179.98
...,...,...
2018262,16.01,-22.50
2018263,15.97,-22.50
2018264,15.93,-22.50


In [36]:
confs = sorted(list(data_dir.iterdir()))

[PosixPath('/projects/inspire/PySAM-MAPS/v1.2/final-backup/01.zarr'),
 PosixPath('/projects/inspire/PySAM-MAPS/v1.2/final-backup/02.zarr'),
 PosixPath('/projects/inspire/PySAM-MAPS/v1.2/final-backup/03.zarr'),
 PosixPath('/projects/inspire/PySAM-MAPS/v1.2/final-backup/04.zarr'),
 PosixPath('/projects/inspire/PySAM-MAPS/v1.2/final-backup/05.zarr'),
 PosixPath('/projects/inspire/PySAM-MAPS/v1.2/final-backup/06.zarr'),
 PosixPath('/projects/inspire/PySAM-MAPS/v1.2/final-backup/07.zarr'),
 PosixPath('/projects/inspire/PySAM-MAPS/v1.2/final-backup/08.zarr'),
 PosixPath('/projects/inspire/PySAM-MAPS/v1.2/final-backup/09.zarr'),
 PosixPath('/projects/inspire/PySAM-MAPS/v1.2/final-backup/10.zarr'),
 PosixPath('/projects/inspire/PySAM-MAPS/v1.2/final-backup/11.zarr')]

In [38]:
import os
import sys
import holoviews as hv

from selenium import webdriver
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.firefox.options import Options
from bokeh.io import export_png, export_svgs

def make_firefox_driver():
    env_bin = os.path.join(sys.prefix, "bin")
    firefox_bin = os.path.join(env_bin, "firefox")
    gecko_bin = os.path.join(env_bin, "geckodriver")

    os.environ["PATH"] = env_bin + os.pathsep + os.environ["PATH"]
    os.environ["TMPDIR"] = os.path.expanduser("~/tmp")
    os.makedirs(os.environ["TMPDIR"], exist_ok=True)

    opts = Options()
    opts.binary_location = firefox_bin
    opts.add_argument("-headless")

    service = Service(
        executable_path=gecko_bin,
        log_output=os.path.expanduser("~/geckodriver.log"),
    )

    return webdriver.Firefox(service=service, options=opts)



In [39]:
import hvplot.xarray  # noqa
import cartopy.crs as ccrs


# optional but strongly recommended:
# compute a shared color scale across all configurations
vmin = np.inf
vmax = -np.inf

for i in range(1, 12):
    conf_data = xr.open_zarr(confs[i])
    vals = conf_data.edgetoedge.mean(dim="time")
    vmin = min(vmin, float(vals.min().compute()))
    vmax = max(vmax, float(vals.max().compute()))

for i in range(1, 12):
    print(f"running conf {i}")
    conf_data = xr.open_zarr(confs[i])

    print("converting gids to lat lon...")
    mean_subset = pvdeg.utilities.gids_dataset_to_coords_dataset(
        conf_data.edgetoedge.mean(dim="time"),
        gids_mapping_df
    )
    
    # lon2d, lat2d = xr.broadcast(mean_subset.longitude, mean_subset.latitude)
    # broadcast in the order of da dims, then force exact dim order match
    lat2d, lon2d = xr.broadcast(mean_subset.latitude, mean_subset.longitude)
    lon2d = lon2d.transpose(*mean_subset.dims)
    lat2d = lat2d.transpose(*mean_subset.dims)
    
    arr = mean_subset.assign_coords(
        longitude_2d=lon2d,
        latitude_2d=lat2d,
    )

    print("plotting configuration")
    xmin, xmax = (float(arr.longitude_2d.min()), float(arr.longitude_2d.max()))
    ymin, ymax = (float(arr.latitude_2d.min()), float(arr.latitude_2d.max()))
    # plot = arr.hvplot.quadmesh(
    #     x="longitude_2d",
    #     y="latitude_2d",
    #     z="edgetoedge",
    #     geo=True,
    #     crs=ccrs.PlateCarree(),
    #     projection=ccrs.AlbersEqualArea(
    #         central_longitude=-96,
    #         central_latitude=37.5,
    #         standard_parallels=(29.5, 45.5),
    #     ),
    #     project=True,
    #     rasterize=True,
    #     coastline=True,
    #     cmap="inferno",
    #     colorbar=True,
    #     width=4000,
    #     height=2400,
    #     xlim=(xmin, xmax),
    #     ylim=(ymin, ymax),
    # )
    plot = arr.hvplot.quadmesh(
        x="longitude_2d",
        y="latitude_2d",
        z="edgetoedge",
        geo=True,
        crs=ccrs.PlateCarree(),
        projection=ccrs.AlbersEqualArea(
            central_longitude=-96,
            central_latitude=37.5,
            standard_parallels=(29.5, 45.5),
        ),
        project=True,
        rasterize=True,
        coastline=True,
        cmap="inferno",
        colorbar=True,
        clabel="Mean edge-to-edge value",
        clim=(vmin, vmax),
        colorbar_position="right",
        colorbar_opts={
            "width": 35,
            "title_standoff": 10,
            "major_label_text_font_size": "18pt",
            "title_text_font_size": "20pt",
        },
        width=4000,
        height=2400,
        xlim=(xmin, xmax),
        ylim=(ymin, ymax),
    )

    driver = make_firefox_driver()
    plot_state = hv.renderer("bokeh").get_plot(plot).state
    
    export_png(
        plot_state,
        filename=f"conf{i}-inferno-fullres.png",
        webdriver=driver,
        timeout=30,
    )
    
    driver.quit()

running conf 1
converting gids to lat lon...
converting gids to lat lon...


/kfs3/scratch/tford/envs/render/lib/python3.13/site-packages/pvdeg/utilities.py:1567: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  stacked = ds_gids.drop(["gid"])


plotting configuration
running conf 2
converting gids to lat lon...
converting gids to lat lon...


/kfs3/scratch/tford/envs/render/lib/python3.13/site-packages/pvdeg/utilities.py:1567: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  stacked = ds_gids.drop(["gid"])


plotting configuration
running conf 3
converting gids to lat lon...
converting gids to lat lon...


/kfs3/scratch/tford/envs/render/lib/python3.13/site-packages/pvdeg/utilities.py:1567: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  stacked = ds_gids.drop(["gid"])


plotting configuration
running conf 4
converting gids to lat lon...
converting gids to lat lon...


/kfs3/scratch/tford/envs/render/lib/python3.13/site-packages/pvdeg/utilities.py:1567: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  stacked = ds_gids.drop(["gid"])


KeyboardInterrupt: 

/kfs3/scratch/tford/envs/render/lib/python3.13/site-packages/pvdeg/utilities.py:1567: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  stacked = ds_gids.drop(["gid"])


In [23]:


xmin, xmax = (float(arr.longitude_2d.min()), float(arr.longitude_2d.max()))
ymin, ymax = (float(arr.latitude_2d.min()), float(arr.latitude_2d.max()))

plot = arr.hvplot.quadmesh(
    x="longitude_2d",
    y="latitude_2d",
    z="edgetoedge",
    geo=True,
    crs=ccrs.PlateCarree(),
    projection=ccrs.AlbersEqualArea(
        central_longitude=-96,
        central_latitude=37.5,
        standard_parallels=(29.5, 45.5),
    ),
    project=True,
    rasterize=True,
    coastline=True,
    cmap="inferno",
    colorbar=True,
    width=4000,
    height=2400,
    xlim=(xmin, xmax),
    ylim=(ymin, ymax),
)

plot

:DynamicMap   []
   :Overlay
      .Image.I     :Image   [longitude_2d,latitude_2d]   (edgetoedge)
      .Coastline.I :Feature   [Longitude,Latitude]